In [1]:
import torch, subprocess, textwrap, os, json, time
print("torch cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    print("mem (GB):", torch.cuda.get_device_properties(0).total_memory/1024**3)
!nvidia-smi

torch cuda available: True
device: NVIDIA A100-SXM4-40GB
capability: (8, 0)
mem (GB): 39.55743408203125
Sun Jan 25 09:35:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             48W /  400W |       5MiB /  40960MiB |      0%      Default |
|                                   

In [2]:
# CÉLULA 1 — Setup (Drive, GPU, caminhos fixos)

from pathlib import Path
import os, sys, json, re, glob
import numpy as np

# (colab) mount drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# GPU
import torch
print("torch cuda available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# caminhos (ajuste aqui se mudar nomes/pastas)
DRIVE = Path("/content/drive/MyDrive")

MOSAIC_DIR = DRIVE / "GEE_Exports"
RF_MAP     = MOSAIC_DIR / "rf_milho_mask1_mt_2023_c10.tif"

RUN_ROOT   = DRIVE / "unet_runs" / "unet_mt2023_v2_run1"
CKPT_BEST  = RUN_ROOT / "checkpoints" / "best.pt"

# saída (predições por tile)
OUT_DIR    = DRIVE / "unet_preds_mt2023_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# sanity
assert MOSAIC_DIR.exists(), f"não achei MOSAIC_DIR: {MOSAIC_DIR}"
assert CKPT_BEST.exists(),  f"não achei CKPT_BEST: {CKPT_BEST}"
print("MOSAIC_DIR:", MOSAIC_DIR)
print("CKPT_BEST :", CKPT_BEST, "| size(MB)=", CKPT_BEST.stat().st_size/1024/1024)
print("RF_MAP exists?", RF_MAP.exists(), "|", RF_MAP)
print("OUT_DIR:", OUT_DIR)

Mounted at /content/drive
torch cuda available: True
device: cuda
MOSAIC_DIR: /content/drive/MyDrive/GEE_Exports
CKPT_BEST : /content/drive/MyDrive/unet_runs/unet_mt2023_v2_run1/checkpoints/best.pt | size(MB)= 88.99758815765381
RF_MAP exists? True | /content/drive/MyDrive/GEE_Exports/rf_milho_mask1_mt_2023_c10.tif
OUT_DIR: /content/drive/MyDrive/unet_preds_mt2023_v1


In [3]:
# CÉLULA 2 — Listar tiles de mosaico + garantir padrão de nomes (169)

import pandas as pd

pat = str(MOSAIC_DIR / "unet_mt_2023_mosaic_x*_y*.tif")
mosaic_files = sorted(glob.glob(pat))

print("mosaic tiles encontrados:", len(mosaic_files))
assert len(mosaic_files) > 0, "não achei tiles de mosaico"
# MT deve ser 169 se bounding box 13x13
# não travo em 169 pq você pode ter excluído/gerado diferente, mas aviso
if len(mosaic_files) != 169:
    print("⚠️ esperado 169 (13x13). encontrado:", len(mosaic_files))

# extrai key x.. y.. do nome
rx = re.compile(r"unet_mt_2023_mosaic_(x-?\d+p\d+)_y(-?\d+p\d+)\.tif$")
rows = []
bad = 0
for fp in mosaic_files:
    name = Path(fp).name
    m = rx.search(name)
    if not m:
        bad += 1
        continue
    x_tag = m.group(1)
    y_tag = "y" + m.group(2) if not m.group(2).startswith("y") else m.group(2)
    key = f"{x_tag}_{y_tag}"  # ex: x-62p00_y-12p00
    rows.append({"key": key, "mosaic_path": fp})

print("tiles com nome ok:", len(rows), "| ruins:", bad)
assert len(rows) > 0 and bad == 0, "há tiles com padrão de nome inesperado"

manifest_tiles = pd.DataFrame(rows).sort_values("key").reset_index(drop=True)
manifest_tiles.head()

mosaic tiles encontrados: 169
tiles com nome ok: 169 | ruins: 0


,key,mosaic_path
0,x-50p00_y-10p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...
1,x-50p00_y-11p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...
2,x-50p00_y-12p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...
3,x-50p00_y-13p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...
4,x-50p00_y-14p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...


In [6]:
# CÉLULA — Modelo EXATO do treino + load best.pt (strict=True)
import torch
import torch.nn as nn

# carrega checkpoint
ckpt = torch.load(CKPT_BEST, map_location="cpu")
sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

# infere in_ch/base do próprio state_dict (robusto)
w = sd["enc1.net.0.weight"]          # (base, in_ch, 3, 3)
base = int(w.shape[0])
in_ch = int(w.shape[1])
print("inferido do ckpt -> in_ch:", in_ch, "| base:", base)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=21, base=32):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base*2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base*2, base*4)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = ConvBlock(base*4, base*8)
        self.pool4 = nn.MaxPool2d(2)

        self.bott = ConvBlock(base*8, base*16)

        self.up4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.dec4 = ConvBlock(base*16, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = ConvBlock(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = ConvBlock(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = ConvBlock(base*2, base)

        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        b  = self.bott(self.pool4(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.head(d1)  # logits (B,1,H,W)

model = UNet(in_ch=in_ch, base=base).to(device)
model.load_state_dict(sd, strict=True)
model.eval()

print("OK ✅ checkpoint carregado em:", device)
print("ckpt epoch:", ckpt.get("epoch", None), "| best_val:", ckpt.get("best_val", None))

inferido do ckpt -> in_ch: 21 | base: 32
OK ✅ checkpoint carregado em: cuda
ckpt epoch: 7 | best_val: 0.05278628298624729


In [7]:
from pathlib import Path
import re, json, math, gc
import numpy as np
import pandas as pd
from tqdm import tqdm

import rasterio
from rasterio.enums import Resampling

import torch

# você já tem:
# - model (carregado e model.eval())
# - device (cuda)
# - MOSAIC_DIR, OUT_DIR, RUN_ROOT, RF_MAP
# - tiles_df com colunas: key, mosaic_path

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PROB = OUT_DIR / "prob_tiles"
OUT_PRED = OUT_DIR / "pred_tiles"
OUT_PROB.mkdir(parents=True, exist_ok=True)
OUT_PRED.mkdir(parents=True, exist_ok=True)

# threshold padrão (use o que você escolheu no notebook anterior; pode mudar depois)
BEST_THR = float(globals().get("BEST_THR", 0.5))
print("BEST_THR:", BEST_THR)

# escala do mosaico
SCALE_U16 = 10000.0

# inferência por blocos (para não explodir RAM/VRAM)
# blocos 1024 geralmente ok na A100; se der OOM reduza para 768 ou 512
BLOCK = 1024
BATCH_SUBTILES = 4  # quantos blocos por forward (aumenta throughput; se OOM, reduza p/ 1-2)

# salvar prob como float16 (COG)
SAVE_PROB = True
SAVE_PRED = True

# compressão
COG_COMPRESS = "LZW"
PRED_NODATA = 255

BEST_THR: 0.5


In [8]:
_key_re = re.compile(r"_x(-?\d+p\d+)_y(-?\d+p\d+)\.tif$")

def key_from_name(fname: str) -> str:
    m = _key_re.search(fname)
    if not m:
        return None
    return f"x{m.group(1)}_y{m.group(2)}".replace("x", "x").replace("y", "y")

def np_to_cog(path_out: Path, arr, profile, dtype=None, nodata=None, compress=COG_COMPRESS):
    prof = profile.copy()
    if dtype is not None:
        prof["dtype"] = dtype
    prof.update(
        driver="GTiff",
        compress=compress,
        tiled=True,
        blockxsize=256,
        blockysize=256,
        interleave="band" if arr.ndim == 3 else "pixel"
    )
    if nodata is not None:
        prof["nodata"] = nodata

    # rasterio espera (bands, H, W)
    if arr.ndim == 2:
        data = arr[np.newaxis, ...]
        prof["count"] = 1
    else:
        data = arr
        prof["count"] = data.shape[0]

    with rasterio.open(path_out, "w", **prof) as dst:
        dst.write(data)

def read_mosaic_u16(path: str):
    with rasterio.open(path) as src:
        prof = src.profile
        # (bands, H, W)
        X = src.read().astype(np.uint16)
    return X, prof

print("helpers OK")

helpers OK


In [9]:
@torch.no_grad()
def infer_tile_blockwise(mosaic_path: str, thr: float = 0.5,
                         block: int = 1024, batch_subtiles: int = 4):
    """
    Retorna:
      prob: (H,W) float16 (0..1) se SAVE_PROB
      pred: (H,W) uint8 {0,1} se SAVE_PRED
    """
    X_u16, prof = read_mosaic_u16(mosaic_path)          # (C,H,W)
    C, H, W = X_u16.shape
    assert C == 21, f"esperava 21 bandas, veio {C} em {mosaic_path}"

    prob_out = np.zeros((H, W), dtype=np.float16) if SAVE_PROB else None
    pred_out = np.zeros((H, W), dtype=np.uint8)  if SAVE_PRED else None

    # grid de janelas
    xs = list(range(0, W, block))
    ys = list(range(0, H, block))

    # pre-aloca listas p/ micro-batch
    tiles_batch = []
    locs_batch = []

    def flush_batch():
        nonlocal tiles_batch, locs_batch
        if not tiles_batch:
            return

        xb = np.stack(tiles_batch, axis=0)  # (B,C,h,w) float32
        xt = torch.from_numpy(xb).to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(xt).squeeze(1)          # (B,h,w)
            p = torch.sigmoid(logits).float()      # (B,h,w) float32

        p_np = p.detach().cpu().numpy()

        for i, (y0, y1, x0, x1) in enumerate(locs_batch):
            pp = p_np[i, : (y1-y0), : (x1-x0)]
            if SAVE_PROB:
                prob_out[y0:y1, x0:x1] = pp.astype(np.float16)
            if SAVE_PRED:
                pred_out[y0:y1, x0:x1] = (pp >= thr).astype(np.uint8)

        # limpa
        tiles_batch = []
        locs_batch = []
        del xt, logits, p
        torch.cuda.empty_cache()

    # percorre janelas
    for y0 in ys:
        for x0 in xs:
            y1 = min(y0 + block, H)
            x1 = min(x0 + block, W)

            # recorte e normaliza
            sub = X_u16[:, y0:y1, x0:x1].astype(np.float32) / SCALE_U16  # (C,h,w)
            tiles_batch.append(sub)
            locs_batch.append((y0, y1, x0, x1))

            if len(tiles_batch) >= batch_subtiles:
                flush_batch()

    flush_batch()

    return prob_out, pred_out, prof

print("infer_tile_blockwise OK")

infer_tile_blockwise OK


In [11]:
from pathlib import Path
import re
import pandas as pd

MOSAIC_DIR = Path(MOSAIC_DIR)

# regex do seu padrão
re_m = re.compile(r"^unet_mt_2023_mosaic_(x-?\d+p\d+)_?(y-?\d+p\d+)\.tif$", re.IGNORECASE)
# no seu caso é "..._x-62p00_y-12p00.tif"
re_m = re.compile(r"^unet_mt_2023_mosaic_(x-?\d+p\d+)_?(y-?\d+p\d+)\.tif$", re.IGNORECASE)

rows = []
bad = []
for p in sorted(MOSAIC_DIR.glob("unet_mt_2023_mosaic_*.tif")):
    m = re_m.match(p.name)
    if not m:
        bad.append(p.name)
        continue
    key = f"{m.group(1)}_{m.group(2)}"  # ex: x-62p00_y-12p00
    rows.append({"key": key, "mosaic_path": str(p)})

tiles_df = pd.DataFrame(rows).sort_values("key").reset_index(drop=True)

print("mosaic tiles encontrados:", len(tiles_df))
print("tiles com nome ok:", len(rows), "| ruins:", len(bad))
if bad:
    print("ex ruins:", bad[:10])

display(tiles_df.head())

mosaic tiles encontrados: 169
tiles com nome ok: 169 | ruins: 0


,key,mosaic_path
0,x-50p00_y-10p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...
1,x-50p00_y-11p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...
2,x-50p00_y-12p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...
3,x-50p00_y-13p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...
4,x-50p00_y-14p00,/content/drive/MyDrive/GEE_Exports/unet_mt_202...


In [13]:
import numpy as np
import rasterio
from rasterio.windows import Window
import torch

# unet geralmente exige múltiplo de 16 (4 downsamples). se teu modelo tiver 5, é 32.
MULT = 16

@torch.inference_mode()
def infer_tile_blockwise(mosaic_path, thr=0.5, block=1024, scale_u16=10000.0, mult=MULT, amp=True):
    """
    Lê o tile (21 bandas uint16), normaliza p/ [0,1], roda UNet por blocos (FCN),
    com padding p/ múltiplo de `mult` e escreve prob/pred no tamanho original.
    Retorna:
      prob (float32 HxW), pred (uint8 HxW)
    """
    model.eval()

    with rasterio.open(mosaic_path) as ds:
        H, W = ds.height, ds.width

        prob_out = np.zeros((H, W), dtype=np.float32)
        pred_out = np.zeros((H, W), dtype=np.uint8)

        for y0 in range(0, H, block):
            h0 = min(block, H - y0)
            for x0 in range(0, W, block):
                w0 = min(block, W - x0)

                win = Window(x0, y0, w0, h0)
                X = ds.read(window=win)  # (C,h0,w0) uint16
                X = X.astype(np.float32) / float(scale_u16)

                # padding p/ múltiplo do mult (bottom/right)
                ph = (mult - (h0 % mult)) % mult
                pw = (mult - (w0 % mult)) % mult
                if ph or pw:
                    X = np.pad(X, ((0,0),(0,ph),(0,pw)), mode="constant", constant_values=0)

                Xt = torch.from_numpy(X).unsqueeze(0).to(device)  # (1,C,h,w)

                with torch.autocast(device_type="cuda", enabled=(amp and device.type=="cuda")):
                    logits = model(Xt)              # (1,1,h,w) ou (1,h,w) dependendo do forward
                    if logits.ndim == 4:
                        logits = logits[:,0,:,:]
                    prob = torch.sigmoid(logits).float().squeeze(0).cpu().numpy()  # (h,w)

                # crop de volta p/ janela original
                prob = prob[:h0, :w0]
                prob_out[y0:y0+h0, x0:x0+w0] = prob
                pred_out[y0:y0+h0, x0:x0+w0] = (prob >= thr).astype(np.uint8)

        return prob_out, pred_out

In [14]:
test_mp = tiles_df.iloc[0]["mosaic_path"]
prob, pred = infer_tile_blockwise(test_mp, thr=BEST_THR, block=BLOCK, scale_u16=SCALE_U16)

print("prob:", prob.shape, prob.dtype, float(prob.min()), float(prob.max()))
print("pred:", pred.shape, pred.dtype, "pct1:", float((pred==1).mean()))

prob: (3712, 3712) float32 2.086162567138672e-06 0.01751708984375
pred: (3712, 3712) uint8 pct1: 0.0


In [15]:
from pathlib import Path
import numpy as np
import torch

DATASET_ROOT = Path("/content/drive/MyDrive/unet_dataset_mt2023_v2/shards")

# pega 1 patch positivo da val (se existir) para ver prob alta
val_shards = sorted((DATASET_ROOT/"val").glob("*.npz"))
assert len(val_shards) > 0, f"sem shards em {DATASET_ROOT/'val'}"

zp = val_shards[0]
with np.load(zp, allow_pickle=True) as z:
    X = z["X"][0].astype(np.float32) / float(SCALE_U16)     # (H,W,C)
    Y = z["Y"][0].astype(np.uint8)                          # (H,W)
X_t = torch.from_numpy(np.transpose(X, (2,0,1))).unsqueeze(0).to(device)

model.eval()
with torch.inference_mode(), torch.autocast(device_type="cuda", enabled=(device.type=="cuda")):
    logits = model(X_t)
    if logits.ndim == 4: logits = logits[:,0,:,:]
    prob = torch.sigmoid(logits).float().squeeze(0).detach().cpu().numpy()

print("patch prob min/max:", float(prob.min()), float(prob.max()))
print("patch pred pct1 @ BEST_THR:", float((prob >= BEST_THR).mean()))
print("Y unique:", {int(k): int(v) for k,v in zip(*np.unique(Y, return_counts=True))})

patch prob min/max: 1.3709068298339844e-06 0.99951171875
patch pred pct1 @ BEST_THR: 0.275299072265625
Y unique: {1: 257, 255: 65279}


In [16]:
from pathlib import Path
import pandas as pd
import re, os

OUT_DIR = Path(OUT_DIR)
OUT_TILES = OUT_DIR / "tiles_pred"
OUT_TILES.mkdir(parents=True, exist_ok=True)

# checa tiles_df
assert "key" in tiles_df.columns and "mosaic_path" in tiles_df.columns
tiles_df = tiles_df.copy()

def key_to_fname(key: str) -> str:
    return f"unet_mt_2023_pred_{key}.tif"

tiles_df["pred_path"] = tiles_df["key"].apply(lambda k: str(OUT_TILES / key_to_fname(k)))

print("OUT_TILES:", OUT_TILES)
print("ex:", tiles_df.iloc[0][["key","mosaic_path","pred_path"]].to_dict())

OUT_TILES: /content/drive/MyDrive/unet_preds_mt2023_v1/tiles_pred
ex: {'key': 'x-50p00_y-10p00', 'mosaic_path': '/content/drive/MyDrive/GEE_Exports/unet_mt_2023_mosaic_x-50p00_y-10p00.tif', 'pred_path': '/content/drive/MyDrive/unet_preds_mt2023_v1/tiles_pred/unet_mt_2023_pred_x-50p00_y-10p00.tif'}


In [17]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from tqdm import tqdm
import time

def write_pred_geotiff(src_mosaic_path: str, dst_pred_path: str, pred_u8: np.ndarray):
    """
    Escreve 1 banda uint8 (0/1) com mesmo CRS/transform do mosaico tile.
    """
    with rasterio.open(src_mosaic_path) as src:
        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="uint8",
            compress="LZW",
            tiled=True,
            blockxsize=512,
            blockysize=512,
            nodata=255  # opcional: fora da máscara depois, a gente põe 255
        )
        dst_path = Path(dst_pred_path)
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(dst_path, "w", **profile) as dst:
            dst.write(pred_u8.astype(np.uint8), 1)

# parâmetros
BLOCK = int(globals().get("BLOCK", 1024))
BEST_THR = float(globals().get("BEST_THR", 0.5))

# loop com resume
t0 = time.time()
done = 0
skipped = 0
stats_rows = []

for _, r in tqdm(tiles_df.iterrows(), total=len(tiles_df), desc="infer+save tiles"):
    mp = r["mosaic_path"]
    outp = r["pred_path"]

    if Path(outp).exists() and Path(outp).stat().st_size > 1024:
        skipped += 1
        continue

    prob, pred = infer_tile_blockwise(mp, thr=BEST_THR, block=BLOCK, scale_u16=SCALE_U16, amp=True)
    # pred é 0/1
    write_pred_geotiff(mp, outp, pred)

    stats_rows.append({
        "key": r["key"],
        "pred_path": outp,
        "pct1": float((pred == 1).mean()),
        "prob_max": float(prob.max()),
        "prob_mean": float(prob.mean())
    })
    done += 1

dt = time.time() - t0
print(f"tiles done={done} skipped={skipped} time_min={dt/60:.1f}")

# salva stats
stats_df = pd.DataFrame(stats_rows)
stats_csv = OUT_DIR / "tiles_pred_stats.csv"
stats_df.to_csv(stats_csv, index=False)
print("stats salvo em:", stats_csv)
display(stats_df.sort_values("pct1", ascending=False).head(10))

infer+save tiles: 100%|██████████| 169/169 [19:44<00:00,  7.01s/it]

tiles done=169 skipped=0 time_min=19.7
stats salvo em: /content/drive/MyDrive/unet_preds_mt2023_v1/tiles_pred_stats.csv


,key,pred_path,pct1,prob_max,prob_mean
82,x-56p00_y-14p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.408962,1.0,0.423362
81,x-56p00_y-13p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.390158,1.0,0.404570
80,x-56p00_y-12p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.303408,1.0,0.324525
94,x-57p00_y-13p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.284153,1.0,0.300947
71,x-55p00_y-16p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.277276,1.0,0.307130
93,x-57p00_y-12p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.276343,1.0,0.298047
42,x-53p00_y-13p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.259334,1.0,0.273511
41,x-53p00_y-12p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.240593,1.0,0.256534
108,x-58p00_y-14p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.239219,1.0,0.263060
95,x-57p00_y-14p00,/content/drive/MyDrive/unet_preds_mt2023_v1/ti...,0.237236,1.0,0.260693


In [19]:
import subprocess, textwrap

cmd = """
apt-get -qq update
apt-get -qq install -y gdal-bin python3-gdal
gdalbuildvrt --version
gdal_translate --version
"""
r = subprocess.run(["bash","-lc", cmd], capture_output=True, text=True)
print("rc:", r.returncode)
print(r.stdout[-2000:])
print(r.stderr[-2000:])
assert r.returncode == 0, "falha ao instalar/rodar GDAL"

rc: 0
Selecting previously unselected package python3-numpy.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../python3-numpy_1%3a1.21.5-1ubuntu22.04.1_amd64.deb ...
Unpacking python3-numpy (1:1.21.5-1ubuntu22.04.1) ...
Selecting previously unselected package python3-gdal.
Preparing to unpack .../python3-gdal_3.8.4+dfsg-1~jammy0_amd64.deb ...
Unpacking python3-gdal (3.8.4+dfsg-1~jammy0) ...
Selecting previou

In [20]:
from pathlib import Path
import subprocess

TILES_DIR = Path("/content/drive/MyDrive/unet_preds_mt2023_v1/tiles_pred")
OUT_ROOT  = Path("/content/drive/MyDrive/unet_preds_mt2023_v1")
OUT_VRT   = OUT_ROOT / "unet_mt2023_pred.vrt"

tiles = sorted(TILES_DIR.glob("unet_mt_2023_pred_*.tif"))
print("tiles:", len(tiles))
assert len(tiles) == 169, "esperado 169 tiles"

lst = OUT_ROOT / "_pred_tiles_list.txt"
lst.write_text("\n".join(str(p) for p in tiles) + "\n", encoding="utf-8")
print("list file:", lst)

cmd = f"gdalbuildvrt -overwrite -input_file_list '{lst}' '{OUT_VRT}'"
print("running:", cmd[:200], "...")
r = subprocess.run(["bash","-lc", cmd], capture_output=True, text=True)
print("rc:", r.returncode)
print(r.stderr[:2000])
assert r.returncode == 0, "gdalbuildvrt falhou"
print("OK VRT:", OUT_VRT)

tiles: 169
list file: /content/drive/MyDrive/unet_preds_mt2023_v1/_pred_tiles_list.txt
running: gdalbuildvrt -overwrite -input_file_list '/content/drive/MyDrive/unet_preds_mt2023_v1/_pred_tiles_list.txt' '/content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred.vrt' ...
rc: 0

OK VRT: /content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred.vrt


In [21]:
import subprocess
from pathlib import Path

OUT_ROOT = Path("/content/drive/MyDrive/unet_preds_mt2023_v1")
OUT_VRT  = OUT_ROOT / "unet_mt2023_pred.vrt"
OUT_TIF  = OUT_ROOT / "unet_mt2023_pred_full.tif"

cmd = f"""
gdal_translate -of GTiff '{OUT_VRT}' '{OUT_TIF}' \
  -co TILED=YES -co COMPRESS=LZW -co BIGTIFF=YES
"""
r = subprocess.run(["bash","-lc", cmd], capture_output=True, text=True)
print("rc:", r.returncode)
print(r.stderr[:2000])
assert r.returncode == 0, "gdal_translate falhou"

print("OK TIF:", OUT_TIF, "size(MB)=", OUT_TIF.stat().st_size/1024/1024)

rc: 0

OK TIF: /content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_full.tif size(MB)= 30.041455268859863


In [23]:
import rasterio
from pathlib import Path

OUT_TIF = Path("/content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_full.tif")

with rasterio.open(OUT_TIF) as ds:
    print("crs:", ds.crs)
    print("shape:", ds.height, ds.width, "count:", ds.count, "dtype:", ds.dtypes)
    print("bounds:", ds.bounds)
    # amostra central
    w = min(1024, ds.width)
    h = min(1024, ds.height)
    win = rasterio.windows.Window(ds.width//2 - w//2, ds.height//2 - h//2, w, h)
    arr = ds.read(1, window=win)
    print("sample unique:", {int(v): int((arr==v).sum()) for v in list(sorted(set(arr.flatten().tolist())))[:10]})

crs: EPSG:4326
shape: 48240 48240 count: 1 dtype: ('uint8',)
bounds: BoundingBox(left=-62.00019377394638, bottom=-19.00017674288359, right=-48.99977498216866, top=-5.999757951105873)
sample unique: {0: 547598, 1: 500978}


In [24]:
import subprocess
from pathlib import Path

VRT_PATH = OUT_DIR / "unet_mt2023_pred.vrt"
MT_PRED_TIF = OUT_DIR / "unet_mt2023_pred_full.tif"

pred_files = sorted(Path(OUT_TILES).glob("*.tif"))
assert len(pred_files) == 169, f"esperado 169 pred tiles, achei {len(pred_files)}"

lst = OUT_DIR / "_pred_tiles_list.txt"
lst.write_text("\n".join(str(p) for p in pred_files) + "\n", encoding="utf-8")

cmd_vrt = f"gdalbuildvrt -overwrite -input_file_list '{lst}' '{VRT_PATH}'"
r = subprocess.run(["bash","-lc", cmd_vrt], capture_output=True, text=True)
print("gdalbuildvrt rc:", r.returncode)
if r.returncode != 0:
    print(r.stderr[:2000]); raise RuntimeError("gdalbuildvrt falhou")

cmd_tif = f"""
gdal_translate -of GTiff '{VRT_PATH}' '{MT_PRED_TIF}' \
  -co COMPRESS=LZW -co TILED=YES -co BLOCKXSIZE=512 -co BLOCKYSIZE=512 -co BIGTIFF=YES
"""
r = subprocess.run(["bash","-lc", cmd_tif], capture_output=True, text=True)
print("gdal_translate rc:", r.returncode)
if r.returncode != 0:
    print(r.stderr[:2000]); raise RuntimeError("gdal_translate falhou")

print("OK ✅", MT_PRED_TIF, "MB=", MT_PRED_TIF.stat().st_size/(1024**2))

gdalbuildvrt rc: 0
gdal_translate rc: 0
OK ✅ /content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_full.tif MB= 23.073798179626465


In [25]:
import rasterio, numpy as np
from pathlib import Path

RF_MAP = Path("/content/drive/MyDrive/GEE_Exports/rf_milho_mask1_mt_2023_c10.tif")
with rasterio.open(RF_MAP) as ds:
    print("RF crs:", ds.crs, "nodata:", ds.nodata, "dtype:", ds.dtypes, "count:", ds.count)
    a = ds.read(1, window=rasterio.windows.Window(0,0,512,512))
    print("sample unique (first 10):", np.unique(a)[:10])

RF crs: EPSG:4326 nodata: None dtype: ('uint8',) count: 1
sample unique (first 10): [0]


In [26]:
from pathlib import Path
import shutil

MT_PRED_TIF = Path(MT_PRED_TIF)
assert MT_PRED_TIF.exists()

FINAL_NO_MASK = OUT_DIR / "unet_mt2023_pred_full_nomask.tif"
if FINAL_NO_MASK.exists():
    FINAL_NO_MASK.unlink()

shutil.copy2(MT_PRED_TIF, FINAL_NO_MASK)
print("OK ✅ final (sem máscara):", FINAL_NO_MASK, "MB=", FINAL_NO_MASK.stat().st_size/(1024**2))


OK ✅ final (sem máscara): /content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_full_nomask.tif MB= 23.073798179626465


In [27]:
import subprocess
from pathlib import Path

FINAL_NO_MASK = Path(OUT_DIR) / "unet_mt2023_pred_full_nomask.tif"
PNG = Path(OUT_DIR) / "unet_mt2023_pred_preview.png"

cmd = f"gdal_translate -of PNG -outsize 10% 10% '{FINAL_NO_MASK}' '{PNG}'"
r = subprocess.run(["bash","-lc", cmd], capture_output=True, text=True)
print("rc:", r.returncode)
if r.returncode != 0:
    print(r.stderr[:2000])
else:
    print("OK ✅ preview:", PNG)

rc: 0
OK ✅ preview: /content/drive/MyDrive/unet_preds_mt2023_v1/unet_mt2023_pred_preview.png
